# Swin + Mahalanobis — **Baseline (No Channel Attention)**

## Purpose
This notebook is a **controlled ablation** of `Proximity_channel_attention_MAHAL_v2.ipynb`.
It is architecturally identical **except that the Squeeze-and-Excitation (SE) Channel Attention block has been removed**.

Run both notebooks on the same dataset and compare their **t-SNE visualisations** to
evaluate whether the channel attention module genuinely produces:
- better-separated Normal / Anomalous clusters, and
- more illumination-invariant feature representations.

## Key differences from the original notebook
| Aspect | Original (with CA) | **This notebook (no CA)** |
|--------|-------------------|---------------------------|
| Model class | `SwinWithChannelAttention` | `SwinWithoutChannelAttention` |
| SE-Block in forward path | ✅ Yes | ❌ Removed |
| Feature extractor class | `SwinFeatureExtractor` | `SwinFeatureExtractorNoCA` |
| Checkpoint file | `best_swin_channel_attention_model.pth` | `best_swin_NO_channel_attention_model.pth` |
| Pickle prefixes | `train_features.pkl`, … | `no_ca_train_features.pkl`, … |
| t-SNE / PCA titles | `Train Set` | `Train Set — No Channel Attention (Baseline)` |
| Mahalanobis CSV | `mahal_predictions.csv` | `no_ca_mahal_predictions.csv` |

## How to compare
After training both models, place the two t-SNE `*_tsne.png` images side by side.
A **higher 2-D centroid distance** and **tighter, less overlapping clusters** in the
with-CA plot provides concrete visual evidence for the illumination-invariance claim.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install timm torch torchvision scikit-learn matplotlib seaborn

In [ ]:
# ============================================================================
# STEP 2: Import Libraries
# ============================================================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
import timm
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ============================================================================
# STEP 3: Set Random Seeds for Reproducibility
# ============================================================================
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [ ]:
# ============================================================================
# STEP 4: Define Dataset Paths
# ============================================================================
# Update these paths to the actual location of your dataset
train_dir = '/content/drive/MyDrive/dataset_S/NIAD-LL/Train'
val_dir = '/content/drive/MyDrive/dataset_S/NIAD-LL/Val'
test_dir = '/content/drive/MyDrive/dataset_S/NIAD-LL/Test'

# Example of how to specify a path if your dataset is in a different structure
# train_dir = '/content/drive/MyDrive/MyDataset/train'
# val_dir = '/content/drive/MyDrive/MyDataset/val'
# test_dir = '/content/drive/MyDrive/MyDataset/test'

# Assuming 'NiAD-large' is a directory containing 'Train', 'Val', and 'Test' subdirectories
# and each of those contains 'normal' and 'anomalous' subdirectories.
# If your structure is different, please update the paths accordingly.

In [ ]:
# ============================================================================
# STEP 5: Channel Attention Module — REMOVED (Baseline / No-Attention Model)
# ============================================================================
# This notebook is a BASELINE that deliberately omits the Squeeze-and-Excitation
# (SE) Channel Attention block.  It is intended to be run alongside
# Proximity_channel_attention_MAHAL_v2.ipynb so that the two t-SNE plots can
# be compared side-by-side to substantiate (or refute) the claim that channel
# attention produces illumination-invariant, better-separated feature spaces.
#
# Changes from the original notebook
# ------------------------------------
#   1. ChannelAttention class removed
#   2. SwinWithoutChannelAttention: Swin → GAP → 1024-d → MLP → logits
#      (no SE-Block in the forward path)
#   3. SwinFeatureExtractorNoCA: mirrors the new forward path
#   4. Model checkpoint saved as  best_swin_NO_channel_attention_model.pth
#   5. All pickle / CSV outputs prefixed with  no_ca_
#   6. t-SNE title updated so plots are unambiguously labelled
print("Baseline model (No Channel Attention) — cell acknowledged.")


In [ ]:
# ============================================================================
# STEP 6: Custom Dataset Class
# ============================================================================
class AnomalyDataset(Dataset):
    """
    Custom Dataset for loading normal and anomalous images
    Expected structure:
        root_dir/
            ├── normal/
            └── anomalous/
    """
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.images = []
        self.labels = []
        self.class_names = ['Normal', 'Anomalous']

        # Load normal images (label 0)
        normal_dir = os.path.join(root_dir, 'Normal')
        if os.path.exists(normal_dir):
            for img_name in os.listdir(normal_dir):
                if img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
                    self.images.append(os.path.join(normal_dir, img_name))
                    self.labels.append(0)

        # Load anomalous images (label 1)
        anomalous_dir = os.path.join(root_dir, 'Anomalous')
        if os.path.exists(anomalous_dir):
            for img_name in os.listdir(anomalous_dir):
                if img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
                    self.images.append(os.path.join(anomalous_dir, img_name))
                    self.labels.append(1)

        print(f"Loaded {len(self.images)} images from {root_dir}")
        print(f"Normal: {self.labels.count(0)}, Anomalous: {self.labels.count(1)}")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")
            # Return a blank image if loading fails
            image = Image.new('RGB', (224, 224))

        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
import torch
import torch.nn as nn
import timm


# ============================================================================
# BASELINE Model: Swin Transformer + MLP Classifier  (NO Channel Attention)
#
# Pipeline:
#   Input frames
#   → Swin Transformer (hierarchical patch merging + transformer blocks)
#   → Global Average Pool  [squeeze W'×H'×C → 1×1×C]
#   → FC Classifier        [→ Predictions]
#
# Channel Attention (SE-Block) is intentionally absent so that t-SNE
# visualisations can be compared against the full model to assess whether
# the SE-Block genuinely improves feature separability.
# ============================================================================
class SwinWithoutChannelAttention(nn.Module):
    def __init__(self, num_classes=2, pretrained=True):
        super().__init__()

        # ── Swin Transformer backbone ──────────────────────────────────────
        self.swin = timm.create_model(
            'swin_base_patch4_window7_224',
            pretrained=pretrained,
            num_classes=0          # raw feature output
        )

        # ── Global Average Pool (squeeze 1×1×C) ───────────────────────────
        self.global_pool = nn.AdaptiveAvgPool2d(1)

        # Swin-Base output channels = 1024
        feature_channels = 1024

        # ── MLP Classifier  (NO SE-Block between pool and classifier) ──────
        self.classifier = nn.Sequential(
            nn.Linear(feature_channels, 512),
            nn.LayerNorm(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),

            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),

            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        feat = self.swin(x)   # → (B, seq_len, C) for Swin

        # Normalise output shape to (B, C)
        if feat.dim() == 3:
            feat = feat.mean(dim=1)
        elif feat.dim() == 4:
            feat = self.global_pool(feat).flatten(1)

        # No SE-Block — go straight to classifier
        return self.classifier(feat)          # (B, num_classes)


In [ ]:
# STEP 8: Define Data Transforms
# ============================================================================
print("\nDefining data transforms...")

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
# ============================================================================
# STEP 9: Load Datasets
# ============================================================================
print("\nLoading datasets...")

train_dataset = AnomalyDataset(train_dir, transform=train_transform)
val_dataset = AnomalyDataset(val_dir, transform=val_test_transform)
test_dataset = AnomalyDataset(test_dir, transform=val_test_transform)


In [ ]:
# ============================================================================
# STEP 10: Create DataLoaders
# ============================================================================
batch_size = 32
num_workers = 2

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True
)

print(f"\nDataset Statistics:")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

In [ ]:
# ============================================================================
# STEP 11: Initialize Model, Loss, and Optimizer
# ============================================================================
print("\nInitializing model...")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# All layers are created in __init__ — no lazy forward-pass build
model = SwinWithoutChannelAttention(num_classes=2, pretrained=True)
model = model.to(device)

# ----------------------------------------------------------------------------
# Selective freezing: keep early Swin stages frozen, unfreeze last 2 stages
# for domain adaptation + always unfreeze channel_attention + classifier.
#
# Why partial unfreeze?
#   - Fully frozen backbone → features are pure ImageNet, not surveillance-domain
#   - Unfreezing layers.2 & layers.3 adapts high-level spatial semantics
#   - Early stages (layers.0, layers.1) stay frozen → preserve low-level filters
# ----------------------------------------------------------------------------

# Step 1: freeze everything
for param in model.parameters():
    param.requires_grad = False

# Step 2: unfreeze Swin last 2 stages
for name, param in model.swin.named_parameters():
    if name.startswith("layers.2") or name.startswith("layers.3"):
        param.requires_grad = True

# Step 3: always unfreeze channel_attention + classifier
for name, param in model.named_parameters():
    if "classifier" in name:
        param.requires_grad = True

# ----------------------------------------------------------------------------
# Report trainable parameter counts
# ----------------------------------------------------------------------------
trainable_params = [name for name, p in model.named_parameters() if p.requires_grad]
total_params     = sum(p.numel() for p in model.parameters())
trainable_count  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable_count:,}  ({100*trainable_count/total_params:.1f}%)")
print(f"Trainable layers    : {len(trainable_params)}")

# ----------------------------------------------------------------------------
# Loss — reduced label_smoothing (0.05 instead of 0.1)
# 0.1 hurts a partially-frozen model that is still finding its footing;
# 0.05 keeps the soft-target benefit without over-penalising correct logits.
# ----------------------------------------------------------------------------
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

# ----------------------------------------------------------------------------
# Optimizer — separate learning-rate groups:
#   • Swin fine-tune layers  → lr=1e-5  (gentle nudge, backbone is fragile)
#   • channel_attention + classifier → lr=1e-4  (head trains at normal speed)
#   • weight_decay 1e-4 (was 5e-3 which over-regularised the tiny head)
# ----------------------------------------------------------------------------
swin_finetune_params = [
    p for n, p in model.named_parameters()
    if p.requires_grad and ("swin.layers.2" in n or "swin.layers.3" in n)
]
head_params = [
    p for n, p in model.named_parameters()
    if p.requires_grad and "classifier" in n
]

optimizer = optim.AdamW([
    {"params": swin_finetune_params, "lr": 1e-5, "weight_decay": 1e-4},
    {"params": head_params,          "lr": 1e-4, "weight_decay": 1e-4},
])

# Scheduler — cosine annealing over 50 epochs
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-7)

print(f"\nOptimizer param groups:")
print(f"  Swin fine-tune  : {len(swin_finetune_params)} tensors  lr=1e-5")
print(f"  Head (clf only) : {len(head_params)} tensors  lr=1e-4")


In [ ]:
# ============================================================================
# STEP 12: Training Function
# ============================================================================
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(dataloader, desc='Training')
    for inputs, labels in pbar:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{100*correct/total:.2f}%'})

    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100 * correct / total
    return epoch_loss, epoch_acc

In [ ]:
# ============================================================================
# STEP 13: Validation Function
# ============================================================================
def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        pbar = tqdm(dataloader, desc='Validation')
        for inputs, labels in pbar:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{100*correct/total:.2f}%'})

    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100 * correct / total
    return epoch_loss, epoch_acc, all_predictions, all_labels

In [ ]:
# ============================================================================
# STEP 14: Training Loop
# ============================================================================
num_epochs = 50
best_val_acc = 0.0
train_losses = []
train_accs = []
val_losses = []
val_accs = []

print("\n" + "="*70)
print("Starting Training...")
print("="*70)

for epoch in range(num_epochs):
    print(f'\nEpoch [{epoch+1}/{num_epochs}]')
    print('-' * 70)

    # Train
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    train_losses.append(train_loss)
    train_accs.append(train_acc)

    # Validate
    val_loss, val_acc, _, _ = validate(model, val_loader, criterion, device)
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    # Print metrics
    print(f'\nTrain Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%')
    print(f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%')
    print(f'Learning Rate: {optimizer.param_groups[0]["lr"]:.2e}')

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'val_loss': val_loss
        }, 'best_swin_NO_channel_attention_model.pth')
        print(f'✓ Best model saved! Val Acc: {val_acc:.2f}%')

    scheduler.step()

    # Early stopping check
    if epoch > 20 and val_acc < best_val_acc - 5:
        print(f"\nEarly stopping triggered. Best Val Acc: {best_val_acc:.2f}%")
        break

print("\n" + "="*70)
print("Training Completed!")
print("="*70)


In [ ]:
# ============================================================================
# STEP 15: Plot Training History
# ============================================================================
print("\nPlotting training history...")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss plot
axes[0].plot(train_losses, label='Train Loss', marker='o', markersize=3)
axes[0].plot(val_losses, label='Val Loss', marker='s', markersize=3)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy plot
axes[1].plot(train_accs, label='Train Acc', marker='o', markersize=3)
axes[1].plot(val_accs, label='Val Acc', marker='s', markersize=3)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================================
# STEP 16: Load Best Model and Evaluate on Test Set
# ============================================================================
print("\n" + "="*70)
print("Testing Best Model...")
print("="*70)

checkpoint = torch.load('best_swin_NO_channel_attention_model.pth')
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best model from epoch {checkpoint['epoch']+1}")

test_loss, test_acc, test_predictions, test_labels = validate(model, test_loader, criterion, device)

print(f'\n{"="*70}')
print(f"TEST RESULTS")
print(f'{"="*70}')
print(f'Test Loss: {test_loss:.4f}')
print(f'Test Accuracy: {test_acc:.2f}%')

In [ ]:
# ============================================================================
# STEP 17: Detailed Classification Metrics
# ============================================================================
print(f'\n{"="*70}')
print("CLASSIFICATION REPORT")
print(f'{"="*70}')
class_names = ['Normal', 'Anomalous']
print(classification_report(test_labels, test_predictions, target_names=class_names, digits=4))

# Precision, Recall, F1 for each class
precision, recall, f1, support = precision_recall_fscore_support(test_labels, test_predictions, average=None)
print(f'\nPer-Class Metrics:')
for i, name in enumerate(class_names):
    print(f'{name:12s} - Precision: {precision[i]:.4f}, Recall: {recall[i]:.4f}, F1: {f1[i]:.4f}, Support: {support[i]}')

# Overall metrics
precision_avg, recall_avg, f1_avg, _ = precision_recall_fscore_support(test_labels, test_predictions, average='macro')
print(f'\nMacro Average - Precision: {precision_avg:.4f}, Recall: {recall_avg:.4f}, F1: {f1_avg:.4f}')


In [ ]:
# ============================================================================
# STEP 18: Confusion Matrix
# ============================================================================
print(f'\n{"="*70}')
print("CONFUSION MATRIX")
print(f'{"="*70}')

cm = confusion_matrix(test_labels, test_predictions)
print(cm)

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'})
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title(f'Confusion Matrix - Test Accuracy: {test_acc:.2f}%')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================================
# STEP 19: Save Final Results
# ============================================================================
results = {
    'best_val_acc': best_val_acc,
    'test_acc': test_acc,
    'test_loss': test_loss,
    'classification_report': classification_report(test_labels, test_predictions, target_names=class_names, output_dict=True),
    'confusion_matrix': cm.tolist(),
    'train_losses': train_losses,
    'train_accs': train_accs,
    'val_losses': val_losses,
    'val_accs': val_accs
}

import json
with open('training_results.json', 'w') as f:
    json.dump(results, f, indent=4)

print("\n" + "="*70)
print("All results saved successfully!")
print("Files created:")
print("  - best_swin_channel_attention_model.pth")
print("  - training_history.png")
print("  - confusion_matrix.png")
print("  - training_results.json")
print("="*70)


In [ ]:
# ============================================================================
# STEP 20: Inference Function for New Images
# ============================================================================
def predict_image(image_path, model, transform, device, class_names=['Normal', 'Anomalous']):
    """
    Predict the class of a single image
    """
    model.eval()

    # Load and preprocess image
    image = Image.open(image_path).convert('RGB')
    image_tensor = transform(image).unsqueeze(0).to(device)

    # Predict
    with torch.no_grad():
        output = model(image_tensor)
        probabilities = torch.softmax(output, dim=1)
        confidence, predicted = torch.max(probabilities, 1)

    predicted_class = class_names[predicted.item()]
    confidence_score = confidence.item() * 100

    return predicted_class, confidence_score

# Example usage:
# predicted_class, confidence = predict_image('path/to/image.jpg', model, val_test_transform, device)
# print(f"Predicted: {predicted_class} (Confidence: {confidence:.2f}%)")

print("\n✓ Training pipeline complete! You can now use predict_image() for inference.")

In [ ]:
# ============================================================================
# STEP 21: Save Model Weights to HDF5 File
# ============================================================================
import h5py

# Define the path to save the HDF5 file
hdf5_save_path = 'weights_no_ca.h5'

try:
    # Save the model state dictionary to a temporary file
    temp_state_dict_path = 'temp_state_dict.pth'
    torch.save(model.state_dict(), temp_state_dict_path)

    # Load the state dictionary from the temporary file and save to HDF5
    with h5py.File(hdf5_save_path, 'w') as f:
        state_dict = torch.load(temp_state_dict_path, map_location=device)
        for key, value in state_dict.items():
            f.create_dataset(key, data=value.cpu().numpy())

    print(f"\nSuccessfully saved model weights to {hdf5_save_path}")

    # Clean up the temporary file
    os.remove(temp_state_dict_path)

except Exception as e:
    print(f"Error saving model weights to HDF5: {e}")

In [ ]:
# ============================================================================
# Feature Extraction from Swin Transformer — NO Channel Attention (Baseline)
# ============================================================================
import torch
import torch.nn as nn
import numpy as np
import pickle
from tqdm import tqdm

# ============================================================================
# STEP 1: Create Feature Extractor from Trained Baseline Model
# ============================================================================
class SwinFeatureExtractorNoCA(nn.Module):
    """
    Extract 256-d features from the baseline model (no SE-Block).
    Forward path: Swin → GAP (if needed) → MLP layers [:-1] → 256-d vector.
    """
    def __init__(self, trained_model):
        super(SwinFeatureExtractorNoCA, self).__init__()

        self.swin        = trained_model.swin
        self.global_pool = trained_model.global_pool

        # All classifier layers except the final Linear(256 → num_classes)
        self.feature_layers = nn.Sequential(
            *list(trained_model.classifier.children())[:-1]
        )

    def forward(self, x):
        features = self.swin(x)

        if features.dim() == 4:      # (B, C, H, W)
            features = self.global_pool(features).flatten(1)
        elif features.dim() == 3:    # (B, seq_len, C)
            features = features.mean(dim=1)

        # No SE-Block — directly into MLP head
        features = self.feature_layers(features)     # → 256-d
        return features

# ============================================================================
# STEP 2–8: Feature extraction, saving, and loading (identical to original)
# ============================================================================
def extract_features_by_class(dataloader, feature_extractor, device):
    feature_extractor.eval()
    normal_features_list    = []
    anomalous_features_list = []

    with torch.no_grad():
        for batch_images, batch_labels in tqdm(dataloader, desc="Extracting features"):
            batch_images  = batch_images.to(device)
            batch_labels  = batch_labels.to(device)
            features      = feature_extractor(batch_images).cpu().numpy()
            labels_np     = batch_labels.cpu().numpy()

            normal_mask    = labels_np == 0
            anomalous_mask = labels_np == 1
            if normal_mask.any():
                normal_features_list.append(features[normal_mask])
            if anomalous_mask.any():
                anomalous_features_list.append(features[anomalous_mask])

    normal_features    = np.concatenate(normal_features_list,    axis=0) if normal_features_list    else np.array([])
    anomalous_features = np.concatenate(anomalous_features_list, axis=0) if anomalous_features_list else np.array([])
    return normal_features, anomalous_features


def save_features_to_pickle(normal_features, anomalous_features, pickle_file):
    data = {
        'normal_features':    normal_features,
        'anomalous_features': anomalous_features,
        'feature_dim':        normal_features.shape[1] if len(normal_features) > 0 else 0
    }
    with open(pickle_file, 'wb') as f:
        pickle.dump(data, f)
    print(f"\n✓ Features saved to: {pickle_file}")
    print(f"  Normal features shape: {normal_features.shape}")
    print(f"  Anomalous features shape: {anomalous_features.shape}")


def load_features_from_pickle(pickle_file):
    with open(pickle_file, 'rb') as f:
        data = pickle.load(f)
    print(f"\n✓ Features loaded from: {pickle_file}")
    print(f"  Normal features shape: {data['normal_features'].shape}")
    print(f"  Anomalous features shape: {data['anomalous_features'].shape}")
    return data['normal_features'], data['anomalous_features']


def extract_and_save_features(model, train_loader, val_loader, test_loader, device, save_dir='./'):
    print("\n" + "="*70)
    print("Creating Baseline Feature Extractor (No Channel Attention)...")
    print("="*70)

    feature_extractor = SwinFeatureExtractorNoCA(model)
    feature_extractor = feature_extractor.to(device)
    feature_extractor.eval()
    print("Feature extractor created. Output dimension: 256")

    for split_name, loader, prefix in [
        ("Training",   train_loader, "no_ca_train"),
        ("Validation", val_loader,   "no_ca_val"),
        ("Test",       test_loader,  "no_ca_test"),
    ]:
        print(f"\n{'='*70}\nExtracting {split_name} Set Features...\n{'='*70}")
        normal, anomalous = extract_features_by_class(loader, feature_extractor, device)
        save_features_to_pickle(normal, anomalous, f'{save_dir}/{prefix}_features.pkl')

    return feature_extractor


# ── Run ──────────────────────────────────────────────────────────────────────
print("\n" + "="*70)
print("BASELINE FEATURE EXTRACTION PIPELINE  (No Channel Attention)")
print("="*70)

checkpoint = torch.load('best_swin_NO_channel_attention_model.pth')
model.load_state_dict(checkpoint['model_state_dict'])
print(f"✓ Baseline model loaded from epoch {checkpoint['epoch']+1}")

feature_extractor = extract_and_save_features(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    device=device,
    save_dir='./'
)

print("\n" + "="*70)
print("Example: Loading Saved Baseline Features")
print("="*70)
test_normal, test_anomalous = load_features_from_pickle('no_ca_test_features.pkl')
print("\nUse these features for t-SNE, PCA, or Mahalanobis to compare with the")
print("channel-attention model's features.")


In [ ]:
# ============================================================================
# t-SNE & PCA — BASELINE (No Channel Attention)
#
# PURPOSE: Compare these plots against the channel-attention model's t-SNE
# to evaluate the claim that the SE-Block produces better-separated,
# illumination-invariant feature representations.
#
# If channel attention truly helps:
#   → This plot should show MORE OVERLAP between Normal/Anomalous clusters
#   → The with-CA plot should show HIGHER 2-D centroid separation
# ============================================================================
# ============================================================================
# Load and Process Extracted Features from Pickle Files
# ============================================================================
import pickle
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from scipy.spatial.distance import cdist

# ============================================================================
# STEP 1: Load Features from Single Pickle File
# ============================================================================
def load_features_single_file(pickle_file):
    """
    Load features from a single pickle file (our format)
    """
    with open(pickle_file, 'rb') as f:
        data = pickle.load(f)

    normal_features = data['normal_features']
    anomalous_features = data['anomalous_features']

    print(f"\n✓ Loaded features from: {pickle_file}")
    print(f"  Normal features shape: {normal_features.shape}")
    print(f"  Anomalous features shape: {anomalous_features.shape}")

    return normal_features, anomalous_features

# ============================================================================
# STEP 2: Load Features from Multiple-Batch Pickle File (Your Format)
# ============================================================================
def load_features_multi_batch(pickle_file):
    """
    Load features from pickle file with multiple batches appended (your TensorFlow format)
    """
    all_normal_features = []
    all_anomalous_features = []

    with open(pickle_file, 'rb') as f:
        while True:
            try:
                data = pickle.load(f)  # Load a batch of data

                # Handle different key names
                if 'positive_features' in data:
                    all_normal_features.append(data['positive_features'])
                elif 'normal_features' in data:
                    all_normal_features.append(data['normal_features'])

                if 'negative_features' in data:
                    all_anomalous_features.append(data['negative_features'])
                elif 'anomalous_features' in data:
                    all_anomalous_features.append(data['anomalous_features'])

            except EOFError:
                break  # Stop when end of file is reached
            except Exception as e:
                print(f"Error loading batch: {e}")
                break

    # Convert the lists into NumPy arrays to combine them
    if all_normal_features:
        all_normal_features = np.concatenate(all_normal_features, axis=0)
    else:
        all_normal_features = np.array([])

    if all_anomalous_features:
        all_anomalous_features = np.concatenate(all_anomalous_features, axis=0)
    else:
        all_anomalous_features = np.array([])

    print(f"\n✓ Loaded features from: {pickle_file}")
    print(f"  Total Normal Features Shape: {all_normal_features.shape}")
    print(f"  Total Anomalous Features Shape: {all_anomalous_features.shape}")

    return all_normal_features, all_anomalous_features

# ============================================================================
# STEP 3: Load All Dataset Features
# ============================================================================
print("="*70)
print("LOADING EXTRACTED FEATURES")
print("="*70)

# Load training features
print("\nLoading Training Features...")
train_normal, train_anomalous = load_features_single_file('no_ca_train_features.pkl')

# Load validation features
print("\nLoading Validation Features...")
val_normal, val_anomalous = load_features_single_file('no_ca_val_features.pkl')

# Load test features
print("\nLoading Test Features...")
test_normal, test_anomalous = load_features_single_file('no_ca_test_features.pkl')

# ============================================================================
# STEP 4: Feature Analysis
# ============================================================================
print("\n" + "="*70)
print("FEATURE ANALYSIS")
print("="*70)

def analyze_features(normal_features, anomalous_features, dataset_name):
    """
    Analyze and compare normal vs anomalous features
    """
    print(f"\n{dataset_name} Dataset:")
    print("-" * 50)

    # Basic statistics
    print(f"Normal samples: {len(normal_features)}")
    print(f"Anomalous samples: {len(anomalous_features)}")
    print(f"Feature dimension: {normal_features.shape[1]}")

    # Mean and std
    print(f"\nNormal - Mean: {normal_features.mean():.4f}, Std: {normal_features.std():.4f}")
    print(f"Anomalous - Mean: {anomalous_features.mean():.4f}, Std: {anomalous_features.std():.4f}")

    # Compute centroids
    normal_centroid = normal_features.mean(axis=0)
    anomalous_centroid = anomalous_features.mean(axis=0)

    # Distance between centroids
    centroid_distance = np.linalg.norm(normal_centroid - anomalous_centroid)
    print(f"\nCentroid Distance: {centroid_distance:.4f}")

    # Average intra-class distances
    # Ensure we don't sample more than available
    n_normal_sample = min(100, len(normal_features))
    n_anomalous_sample = min(100, len(anomalous_features))

    normal_distances = cdist(normal_features[:n_normal_sample], [normal_centroid], metric='euclidean').mean() if n_normal_sample > 0 else 0
    anomalous_distances = cdist(anomalous_features[:n_anomalous_sample], [anomalous_centroid], metric='euclidean').mean() if n_anomalous_sample > 0 else 0


    print(f"Avg Normal Distance to Centroid: {normal_distances:.4f}")
    print(f"Avg Anomalous Distance to Centroid: {anomalous_distances:.4f}")

    # Separability score
    # Avoid division by zero if both distances are zero
    separability = centroid_distance / (normal_distances + anomalous_distances) if (normal_distances + anomalous_distances) > 0 else np.inf
    print(f"Separability Score: {separability:.4f} (higher is better)")

    return {
        'centroid_distance': centroid_distance,
        'separability': separability,
        'normal_centroid': normal_centroid,
        'anomalous_centroid': anomalous_centroid
    }

# Analyze all datasets
train_stats = analyze_features(train_normal, train_anomalous, "Training")
val_stats = analyze_features(val_normal, val_anomalous, "Validation")
test_stats = analyze_features(test_normal, test_anomalous, "Test")

# ============================================================================
# STEP 5: Visualize Feature Distributions
# ============================================================================
print("\n" + "="*70)
print("VISUALIZING FEATURE DISTRIBUTIONS")
print("="*70)

def plot_feature_distributions(normal_features, anomalous_features, title):
    """
    Plot feature distributions for normal vs anomalous
    """
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle(title, fontsize=16, fontweight='bold')

    # 1. Mean feature values
    axes[0, 0].plot(normal_features.mean(axis=0), label='Normal', alpha=0.7)
    axes[0, 0].plot(anomalous_features.mean(axis=0), label='Anomalous', alpha=0.7)
    axes[0, 0].set_xlabel('Feature Index')
    axes[0, 0].set_ylabel('Mean Value')
    axes[0, 0].set_title('Mean Feature Values')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    # 2. Std feature values
    axes[0, 1].plot(normal_features.std(axis=0), label='Normal', alpha=0.7)
    axes[0, 1].plot(anomalous_features.std(axis=0), label='Anomalous', alpha=0.7)
    axes[0, 1].set_xlabel('Feature Index')
    axes[0, 1].set_ylabel('Std Value')
    axes[0, 1].set_title('Feature Standard Deviations')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    # 3. Distance to centroid histogram
    normal_centroid = normal_features.mean(axis=0)
    anomalous_centroid = anomalous_features.mean(axis=0)

    normal_dists = np.linalg.norm(normal_features - normal_centroid, axis=1)
    anomalous_dists = np.linalg.norm(anomalous_features - anomalous_centroid, axis=1)

    axes[1, 0].hist(normal_dists, bins=50, alpha=0.6, label='Normal', color='blue')
    axes[1, 0].hist(anomalous_dists, bins=50, alpha=0.6, label='Anomalous', color='red')
    axes[1, 0].set_xlabel('Distance to Class Centroid')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 0].set_title('Distance Distribution to Class Centroids')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    # 4. Feature correlation heatmap (sample)
    sample_normal = normal_features[:min(1000, len(normal_features))]
    sample_anomalous = anomalous_features[:min(1000, len(anomalous_features))]
    all_samples = np.vstack([sample_normal, sample_anomalous])

    # Compute correlation matrix for first 50 features
    corr_matrix = np.corrcoef(all_samples[:, :min(50, all_samples.shape[1])].T)

    sns.heatmap(corr_matrix, ax=axes[1, 1], cmap='coolwarm', center=0,
                cbar_kws={'label': 'Correlation'})
    axes[1, 1].set_title('Feature Correlation (First 50 Features)')

    plt.tight_layout()
    plt.savefig(f'{title.replace(" ", "_").lower()}_feature_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()

# Plot for test set
plot_feature_distributions(test_normal, test_anomalous, "Test Set Feature Analysis")

# ============================================================================
# STEP 6: t-SNE Visualization  ← runs on TRAIN SET (not test)
#
# Why train set?
#   The Gaussian model (Mahalanobis) is FIT on train features.
#   Visualising train features tells you whether the Gaussian assumption
#   is valid BEFORE you use it to classify.  High 2-D centroid separation
#   → Mahalanobis will generalise well.  Low separation → need more tuning.
# ============================================================================
print("
" + "="*70)
print("CREATING t-SNE VISUALIZATION  (train set)")
print("="*70)

def visualize_tsne(normal_features, anomalous_features, title, n_samples=2000):
    """
    t-SNE of features with:
      • Class scatter (blue=Normal, red=Anomalous)
      • Class centroid markers (★) so you can judge separation at a glance
      • KDE density contours to show cluster tightness
    """
    from sklearn.manifold import TSNE
    import matplotlib.pyplot as plt
    import matplotlib.patheffects as pe

    n_normal    = min(n_samples, len(normal_features))
    n_anomalous = min(n_samples, len(anomalous_features))
    if min(n_normal, n_anomalous) < 2:
        print(f"Not enough samples for t-SNE in {title}.")
        return None

    rng = np.random.default_rng(42)
    normal_sample    = normal_features[rng.choice(len(normal_features),    n_normal,    replace=False)]
    anomalous_sample = anomalous_features[rng.choice(len(anomalous_features), n_anomalous, replace=False)]

    all_feat = np.vstack([normal_sample, anomalous_sample])
    labels   = np.array([0]*n_normal + [1]*n_anomalous)

    perplexity = min(30, max(5, len(all_feat) // 10))
    print(f"Running t-SNE on {len(all_feat)} samples  (perplexity={perplexity})...")

    tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity, n_iter=1000)
    feat_2d = tsne.fit_transform(all_feat)

    # ── centroids in 2-D embedding space ──────────────────────────────────
    normal_2d    = feat_2d[labels == 0]
    anomalous_2d = feat_2d[labels == 1]
    c_normal    = normal_2d.mean(axis=0)
    c_anomalous = anomalous_2d.mean(axis=0)

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    fig.suptitle(f"t-SNE — {title}", fontsize=15, fontweight="bold")

    # ── LEFT: scatter + centroid stars ────────────────────────────────────
    ax = axes[0]
    ax.scatter(normal_2d[:, 0],    normal_2d[:, 1],    c="steelblue", alpha=0.5, s=15, label="Normal")
    ax.scatter(anomalous_2d[:, 0], anomalous_2d[:, 1], c="tomato",    alpha=0.5, s=15, label="Anomalous")

    star_kw = dict(marker="*", s=350, zorder=5,
                   path_effects=[pe.withStroke(linewidth=2, foreground="white")])
    ax.scatter(*c_normal,    c="navy",    label="Normal centroid ★",    **star_kw)
    ax.scatter(*c_anomalous, c="darkred", label="Anomalous centroid ★", **star_kw)

    # annotate centroid Euclidean distance in 2-D
    dist_2d = np.linalg.norm(c_normal - c_anomalous)
    ax.set_title(f"Scatter + Centroids  |  2-D centroid dist = {dist_2d:.1f}", fontsize=11)
    ax.set_xlabel("t-SNE dim 1"); ax.set_ylabel("t-SNE dim 2")
    ax.legend(fontsize=9); ax.grid(True, alpha=0.25)

    # ── RIGHT: KDE density contours ──────────────────────────────────────
    ax2 = axes[1]
    ax2.scatter(normal_2d[:, 0],    normal_2d[:, 1],    c="steelblue", alpha=0.3, s=10)
    ax2.scatter(anomalous_2d[:, 0], anomalous_2d[:, 1], c="tomato",    alpha=0.3, s=10)

    try:
        from scipy.stats import gaussian_kde
        for data, color, name in [(normal_2d, "blue", "Normal"),
                                   (anomalous_2d, "red",  "Anomalous")]:
            if len(data) > 10:
                kde = gaussian_kde(data.T, bw_method="scott")
                xg  = np.linspace(feat_2d[:, 0].min()-1, feat_2d[:, 0].max()+1, 120)
                yg  = np.linspace(feat_2d[:, 1].min()-1, feat_2d[:, 1].max()+1, 120)
                Xg, Yg = np.meshgrid(xg, yg)
                Zg = kde(np.vstack([Xg.ravel(), Yg.ravel()])).reshape(Xg.shape)
                ax2.contour(Xg, Yg, Zg, levels=5, colors=color, alpha=0.6, linewidths=1.2)
    except Exception:
        pass   # skip KDE if it errors (e.g., singular covariance)

    # centroid stars on density plot too
    ax2.scatter(*c_normal,    c="navy",    **star_kw)
    ax2.scatter(*c_anomalous, c="darkred", **star_kw)
    ax2.set_title("KDE Density Contours", fontsize=11)
    ax2.set_xlabel("t-SNE dim 1"); ax2.set_ylabel("t-SNE dim 2")
    ax2.grid(True, alpha=0.25)

    # ── Interpretation guide printed to console ───────────────────────────
    print(f"
t-SNE 2-D centroid distance = {dist_2d:.2f}")
    print("  Interpretation:")
    if dist_2d > 20:
        print("  ✓ HIGH separation → Gaussian / Mahalanobis model should generalise well.")
    elif dist_2d > 8:
        print("  ~ MODERATE separation → Mahalanobis may work; watch per-class recall.")
    else:
        print("  ✗ LOW separation → features are not yet discriminative enough.")
        print("    Consider unfreezing more Swin stages or training longer.")

    plt.tight_layout()
    save_name = f'{title.lower().replace(" ", "_")}_tsne.png'
    plt.savefig(save_name, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"✓ Saved → {save_name}")
    return feat_2d

# ── Run t-SNE on TRAIN features (correct for Gaussian model validation) ──
print("
[t-SNE] Using TRAIN SET — these are the features the Gaussian is fitted on.")
train_tsne_2d = visualize_tsne(train_normal, train_anomalous, "Train Set — No Channel Attention (Baseline)")

# Optional: also visualise test set for comparison
visualize_tsne(test_normal, test_anomalous, "Test Set — No Channel Attention (Baseline)")

# ============================================================================
# STEP 7: PCA Visualization
# ============================================================================
print("\n" + "="*70)
print("CREATING PCA VISUALIZATION")
print("="*70)

def visualize_pca(normal_features, anomalous_features, title):
    """
    Create PCA visualization of features
    """
    # Combine features
    all_features = np.vstack([normal_features, anomalous_features])
    labels = np.array([0]*len(normal_features) + [1]*len(anomalous_features))

    if len(all_features) < 2:
        print(f"Not enough samples for PCA visualization in {title}.")
        return

    print(f"Running PCA on {len(all_features)} samples from {title}...")

    # Run PCA
    pca = PCA(n_components=2)
    features_2d = pca.fit_transform(all_features)

    print(f"Explained variance: {pca.explained_variance_ratio_[0]:.4f}, {pca.explained_variance_ratio_[1]:.4f}")
    print(f"Total explained variance: {pca.explained_variance_ratio_.sum():.4f}")

    # Plot
    plt.figure(figsize=(12, 8))
    scatter = plt.scatter(features_2d[labels==0, 0], features_2d[labels==0, 1],
                         c='blue', alpha=0.6, s=20, label='Normal')
    scatter = plt.scatter(features_2d[labels==1, 0], features_2d[labels==1, 1],
                         c='red', alpha=0.6, s=20, label='Anomalous')

    plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)', fontsize=12)
    plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)', fontsize=12)
    plt.title(f'PCA Visualization - {title}', fontsize=14, fontweight='bold')
    plt.legend(fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{title.replace(" ", "_").lower()}_pca.png', dpi=300, bbox_inches='tight')
    plt.show()

    print("✓ PCA visualization complete!")

# Create PCA for test set
visualize_pca(test_normal, test_anomalous, "Test Set — No Channel Attention (Baseline)")

# ============================================================================
# STEP 8: Save Combined Features for Further Analysis
# ============================================================================
print("\n" + "="*70)
print("SAVING COMBINED FEATURES")
print("="*70)

# Combine all features
all_features = {
    'train_normal': train_normal,
    'train_anomalous': train_anomalous,
    'val_normal': val_normal,
    'val_anomalous': val_anomalous,
    'test_normal': test_normal,
    'test_anomalous': test_anomalous,
    'train_stats': train_stats,
    'val_stats': val_stats,
    'test_stats': test_stats
}

with open('no_ca_all_extracted_features.pkl', 'wb') as f:
    pickle.dump(all_features, f)

print("✓ All features saved to: all_extracted_features.pkl")

print("\n" + "="*70)
print("FEATURE LOADING AND ANALYSIS COMPLETE!")
print("="*70)


In [ ]:
import os
import pandas as pd
from pathlib import Path

def create_csv_from_folders(base_dir, output_csv):
    """
    Create CSV file with image paths and labels
    Normal folder -> label 0
    Anomalous folder -> label 1
    """
    data = []

    # Process normal folder (label 0)
    normal_dir = os.path.join(base_dir, 'Normal') # Corrected folder name
    if os.path.exists(normal_dir):
        print(f"Processing normal folder: {normal_dir}")
        normal_images = [f for f in os.listdir(normal_dir)
                        if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff'))]

        for img_name in normal_images:
            img_path = os.path.join(normal_dir, img_name)
            data.append({
                'imagepath': img_path,
                'Groundtruth': 0  # Normal = 0
            })
        print(f"  Found {len(normal_images)} normal images")
    else:
        print(f"Warning: Normal folder not found at {normal_dir}")

    # Process anomalous folder (label 1)
    anomalous_dir = os.path.join(base_dir, 'Anomalous') # Corrected folder name
    if os.path.exists(anomalous_dir):
        print(f"Processing anomalous folder: {anomalous_dir}")
        anomalous_images = [f for f in os.listdir(anomalous_dir)
                           if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff'))]

        for img_name in anomalous_images:
            img_path = os.path.join(anomalous_dir, img_name)
            data.append({
                'imagepath': img_path,
                'Groundtruth': 1  # Anomalous = 1
            })
        print(f"  Found {len(anomalous_images)} anomalous images")
    else:
        print(f"Warning: Anomalous folder not found at {anomalous_dir}")

    # Create DataFrame
    df = pd.DataFrame(data)

    # Save to CSV
    df.to_csv(output_csv, index=False)

    print(f"\n✓ CSV created successfully!")
    print(f"  Total images: {len(df)}")
    # Only try to access 'Groundtruth' if the DataFrame is not empty
    if not df.empty:
        print(f"  Normal (0): {(df['Groundtruth'] == 0).sum()}")
        print(f"  Anomalous (1): {(df['Groundtruth'] == 1).sum()}")
    print(f"  Saved to: {output_csv}")

    # Show first few rows
    if not df.empty:
        print(f"\nFirst 5 rows:")
        print(df.head())
    else:
        print("\nDataFrame is empty, no rows to display.")


    return df

# Specify your base directory containing 'normal' and 'anomalous' folders
base_dir = '/content/drive/MyDrive/dataset_S/Target_dataset/Test'
output_csv = '/content/drive/MyDrive/dataset_S/Target_dataset/corrected.csv'

# Create the CSV
df = create_csv_from_folders(base_dir, output_csv)

In [ ]:
# ============================================================================
# PROXIMITY-BASED PREDICTION USING MAHALANOBIS DISTANCE
# ============================================================================
# Pipeline:
#   1. Load train features (Normal + Anomalous) from pickle
#   2. Fit a full-covariance Gaussian per class on train features
#   3. Load test features in batch via DataLoader (no per-image loop)
#   4. Compute scipy Mahalanobis distance to each class centroid
#   5. Predict: class with the smaller distance wins
#   6. Evaluate with sklearn (classification_report, confusion_matrix)
#   7. Plot seaborn confusion matrix heatmap + save results CSV
# ============================================================================

import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

from scipy.spatial.distance import mahalanobis as scipy_mahalanobis
from scipy.linalg import pinvh
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
)
from tqdm import tqdm

# ── Config ───────────────────────────────────────────────────────────────────
CLASS_NAMES  = ['Normal', 'Anomalous']
REG          = 1e-5          # Tikhonov regularisation added to cov diagonal
SAVE_CSV     = '/content/drive/MyDrive/dataset_S/Target_dataset/no_ca_mahal_predictions.csv'
SAVE_FIG_CM  = 'no_ca_confusion_matrix_mahal.png'
SAVE_FIG_DIST = 'no_ca_mahal_score_distribution.png'

# ── Step 1: Load train features ───────────────────────────────────────────────
print("=" * 70)
print("STEP 1: Loading train features from pickle")
print("=" * 70)
train_normal, train_anomalous = load_features_from_pickle('no_ca_train_features.pkl')
# shape: (N_normal, 256) and (N_anomalous, 256)
print(f"  Train Normal    : {train_normal.shape}")
print(f"  Train Anomalous : {train_anomalous.shape}")

# ── Step 2: Fit full-covariance Gaussian per class ───────────────────────────
print("\n" + "=" * 70)
print("STEP 2: Fitting full-covariance Gaussians on train features")
print("=" * 70)

def fit_gaussian(features, reg=REG):
    """
    Compute mean and regularised inverse covariance for a class.

    Parameters
    ----------
    features : (N, D) numpy array
    reg      : float — added to diagonal before inversion

    Returns
    -------
    mu : (D,)    — class mean
    VI : (D, D)  — inverse of regularised covariance (via pinvh)
    """
    mu  = features.mean(axis=0)                          # (D,)
    cov = np.cov(features, rowvar=False)                 # (D, D)
    cov += reg * np.eye(cov.shape[0])                    # regularise
    VI  = pinvh(cov)                                     # stable symmetric pseudo-inverse
    return mu, VI

mu_normal,    VI_normal    = fit_gaussian(train_normal)
mu_anomalous, VI_anomalous = fit_gaussian(train_anomalous)

print(f"  Normal    — mu.norm={np.linalg.norm(mu_normal):.4f}  "
      f"VI.shape={VI_normal.shape}")
print(f"  Anomalous — mu.norm={np.linalg.norm(mu_anomalous):.4f}  "
      f"VI.shape={VI_anomalous.shape}")

# ── Step 3: Batch-extract test features via DataLoader ───────────────────────
print("\n" + "=" * 70)
print("STEP 3: Extracting test features in batch")
print("=" * 70)

# Use the already-created feature_extractor (SwinFeatureExtractor)
# and test_loader (val_test_transform, no augmentation)
feature_extractor.eval()

all_features   = []
all_true_labels = []

with torch.no_grad():
    for batch_imgs, batch_labels in tqdm(test_loader, desc="Extracting test features"):
        batch_imgs = batch_imgs.to(device)
        feats      = feature_extractor(batch_imgs)          # (B, 256)
        all_features.append(feats.cpu().numpy())
        all_true_labels.extend(batch_labels.numpy().tolist())

test_features  = np.concatenate(all_features, axis=0)      # (N_test, 256)
true_labels    = np.array(all_true_labels)                  # (N_test,)
print(f"  Test features shape : {test_features.shape}")
print(f"  True labels shape   : {true_labels.shape}  "
      f"(Normal={( true_labels==0).sum()}  Anomalous={(true_labels==1).sum()})")

# ── Step 4: Compute Mahalanobis distances ─────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 4: Computing scipy Mahalanobis distances")
print("=" * 70)

# Vectorised — one scipy call per sample (fast numpy inner loop)
d_normal    = np.array([scipy_mahalanobis(x, mu_normal,    VI_normal)
                         for x in tqdm(test_features, desc="  dist→Normal")])
d_anomalous = np.array([scipy_mahalanobis(x, mu_anomalous, VI_anomalous)
                         for x in tqdm(test_features, desc="  dist→Anomalous")])

print(f"  d_normal    — mean={d_normal.mean():.4f}  std={d_normal.std():.4f}")
print(f"  d_anomalous — mean={d_anomalous.mean():.4f}  std={d_anomalous.std():.4f}")

# ── Step 5: Predict ───────────────────────────────────────────────────────────
# Assign the class whose centroid is CLOSER (smaller Mahalanobis distance)
predicted_labels = (d_anomalous <= d_normal).astype(int)   # 1=Anomalous, 0=Normal

# Anomaly score: normalised distance ratio (useful for ROC-AUC)
anomaly_score = d_anomalous / (d_normal + d_anomalous + 1e-8)

# ── Step 6: Evaluate with sklearn ─────────────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 5: Evaluation — sklearn metrics")
print("=" * 70)

acc = accuracy_score(true_labels, predicted_labels)
print(f"\nAccuracy : {acc * 100:.4f}%")

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels,
                             target_names=CLASS_NAMES, digits=4))

precision, recall, f1, support = precision_recall_fscore_support(
    true_labels, predicted_labels, average=None)
print("Per-Class Metrics:")
for i, name in enumerate(CLASS_NAMES):
    print(f"  {name:12s} — Precision: {precision[i]:.4f}  "
          f"Recall: {recall[i]:.4f}  F1: {f1[i]:.4f}  "
          f"Support: {support[i]}")

p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
    true_labels, predicted_labels, average='macro')
print(f"\nMacro Average — Precision: {p_macro:.4f}  "
      f"Recall: {r_macro:.4f}  F1: {f1_macro:.4f}")

try:
    auc = roc_auc_score(true_labels, anomaly_score)
    print(f"ROC-AUC        : {auc:.4f}")
except Exception as e:
    print(f"ROC-AUC        : could not compute ({e})")

# ── Step 7a: Confusion matrix — seaborn heatmap ───────────────────────────────
print("\n" + "=" * 70)
print("STEP 6: Confusion Matrix")
print("=" * 70)

cm = confusion_matrix(true_labels, predicted_labels)
print(cm)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=0.5, ax=axes[0],
            cbar_kws={'label': 'Count'})
axes[0].set_xlabel('Predicted Label', fontsize=12)
axes[0].set_ylabel('True Label',      fontsize=12)
axes[0].set_title(f'Confusion Matrix\n(Mahalanobis)  Acc={acc*100:.2f}%',
                  fontsize=13, fontweight='bold')

# Right: row-normalised (recall per class)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=0.5, vmin=0, vmax=1, ax=axes[1],
            cbar_kws={'label': 'Recall (row-normalised)'})
axes[1].set_xlabel('Predicted Label', fontsize=12)
axes[1].set_ylabel('True Label',      fontsize=12)
axes[1].set_title('Normalised Confusion Matrix\n(row = recall per class)',
                  fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig(SAVE_FIG_CM, dpi=300, bbox_inches='tight')
plt.show()
print(f"  Saved → {SAVE_FIG_CM}")

# ── Step 7b: Anomaly score distribution ───────────────────────────────────────
plt.figure(figsize=(9, 4))
for cls, name, color in [(0, 'Normal', 'steelblue'), (1, 'Anomalous', 'tomato')]:
    mask = true_labels == cls
    plt.hist(anomaly_score[mask], bins=60, alpha=0.65, density=True,
             label=f'{name} (n={mask.sum()})', color=color)
plt.axvline(0.5, color='black', linestyle='--', alpha=0.6, label='Threshold = 0.5')
plt.xlabel('Anomaly Score  d_anomalous / (d_normal + d_anomalous)', fontsize=11)
plt.ylabel('Density', fontsize=11)
plt.title('Mahalanobis Anomaly Score Distribution — Test Set', fontsize=13,
          fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(SAVE_FIG_DIST, dpi=300, bbox_inches='tight')
plt.show()
print(f"  Saved → {SAVE_FIG_DIST}")

# ── Step 8: Save results to CSV ───────────────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 7: Saving results to CSV")
print("=" * 70)

results_df = pd.DataFrame({
    'Groundtruth'         : true_labels,
    'Predicted'           : predicted_labels,
    'AnomalyScore'        : anomaly_score,
    'MahalDistNormal'     : d_normal,
    'MahalDistAnomalous'  : d_anomalous,
})
results_df.to_csv(SAVE_CSV, index=False)
print(f"  Saved → {SAVE_CSV}")
print(f"  Rows  : {len(results_df)}")
print(f"\n✓ Mahalanobis proximity pipeline complete!")
print(f"  Accuracy  : {acc*100:.4f}%")
print(f"  Macro-F1  : {f1_macro:.4f}")
